In [1]:
%load_ext autoreload
%autoreload 2

# PPMI Genetic Data Investigation

## Objective

Investigate genetic predictors of phenoconversion to Parkinson's disease (PD)
in the predefined PPMI at-risk/prodromal cohort.

The genetic data come from two PPMI files:

1. `PPMI_Project_9001...`
   - Polygenic risk scores (PRS)
   - Genetic principal components (PCs)
   - Inferred genetic population / ancestry

2. `iu_genetic_consensus...`
   - Individual genetic variant information
   - Major PD-related genetic loci/variants

The predefined at-risk cohort and phenoconversion outcome are obtained from:

`ppmi_outcomes.csv`

## Main goals

- Define the analytic at-risk cohort
- Define baseline-to-follow-up phenoconversion
- Determine genetic data availability
- Investigate PRS and genetic PCs
- Investigate individual genetic variants
- Assess carrier frequencies
- Identify sparse variables and possible complete/quasi-complete separation
- Investigate redundancy and correlations among genetic predictors
- Construct a defensible genetic feature set for subsequent multimodal prediction

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# ---------------------------------------------------------------------
# Project configuration
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root: {PROJECT_ROOT}")

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected to find 'pyproject.toml' in the project directory."
    )

print("Project root successfully identified.")

Project root: c:\Users\eademola\pd-risk-prediction
Project root successfully identified.


In [4]:
# ---------------------------------------------------------------------
# Input data paths
# ---------------------------------------------------------------------

GENETICS_PRS_PATH = (
    PROJECT_ROOT
    / "data/raw/ppmi/PPMI_Project_9001_20250624_28Jul2026.csv"
)

GENETICS_VARIANTS_PATH = (
    PROJECT_ROOT
    / "data/raw/ppmi/iu_genetic_consensus_20251025_28Jul2026.csv"
)

OUTCOMES_PATH = (
    PROJECT_ROOT
    / "data/processed/outcomes/ppmi_outcomes.csv"
)

print("Input files configured:")
print(f"  PRS/PC data:       {GENETICS_PRS_PATH}")
print(f"  Variant data:      {GENETICS_VARIANTS_PATH}")
print(f"  Outcome data:      {OUTCOMES_PATH}")

Input files configured:
  PRS/PC data:       c:\Users\eademola\pd-risk-prediction\data\raw\ppmi\PPMI_Project_9001_20250624_28Jul2026.csv
  Variant data:      c:\Users\eademola\pd-risk-prediction\data\raw\ppmi\iu_genetic_consensus_20251025_28Jul2026.csv
  Outcome data:      c:\Users\eademola\pd-risk-prediction\data\processed\outcomes\ppmi_outcomes.csv


In [5]:
# ---------------------------------------------------------------------
# Load datasets
# ---------------------------------------------------------------------

genetics_prs = pd.read_csv(GENETICS_PRS_PATH)
genetics_variants = pd.read_csv(GENETICS_VARIANTS_PATH)
outcomes = pd.read_csv(OUTCOMES_PATH)

print("Datasets loaded successfully.")
print()
print(f"PRS/PC dataset:      {genetics_prs.shape[0]:,} rows × {genetics_prs.shape[1]:,} columns")
print(f"Variant dataset:     {genetics_variants.shape[0]:,} rows × {genetics_variants.shape[1]:,} columns")
print(f"Outcome dataset:     {outcomes.shape[0]:,} rows × {outcomes.shape[1]:,} columns")

Datasets loaded successfully.

PRS/PC dataset:      1,319 rows × 109 columns
Variant dataset:     6,265 rows × 21 columns
Outcome dataset:     2,247 rows × 20 columns


## 5. Initial data checks

Check participant identifiers, duplicate records, missingness, and the
structure of the outcome dataset before defining the analytic cohort.

In [6]:
# ---------------------------------------------------------------------
# Initial structural checks
# ---------------------------------------------------------------------

print("Unique participant counts:")
print(f"  PRS/PC dataset:  {genetics_prs['PATNO'].nunique():,}")
print(f"  Variant dataset: {genetics_variants['PATNO'].nunique():,}")
print(f"  Outcome dataset: {outcomes['PATNO'].nunique():,}")

print("\nDuplicate PATNO checks:")

print(
    f"  PRS/PC dataset:  "
    f"{genetics_prs['PATNO'].duplicated().sum():,} duplicate rows"
)

print(
    f"  Variant dataset: "
    f"{genetics_variants['PATNO'].duplicated().sum():,} duplicate rows"
)

print(
    f"  Outcome dataset: "
    f"{outcomes['PATNO'].duplicated().sum():,} duplicate rows"
)

Unique participant counts:
  PRS/PC dataset:  1,319
  Variant dataset: 6,265
  Outcome dataset: 2,247

Duplicate PATNO checks:
  PRS/PC dataset:  0 duplicate rows
  Variant dataset: 0 duplicate rows
  Outcome dataset: 0 duplicate rows


## Define at-risk cohort

In [7]:
# ---------------------------------------------------------------------
# Define the analytic at-risk cohort
# ---------------------------------------------------------------------

at_risk_patno = outcomes["PATNO"].dropna().unique()

print(
    f"At-risk cohort: {len(at_risk_patno):,} unique participants"
)

outcome_duplicates = outcomes["PATNO"].duplicated().sum()

print(
    f"Duplicate PATNO rows in outcomes dataset: "
    f"{outcome_duplicates:,}"
)

At-risk cohort: 2,247 unique participants
Duplicate PATNO rows in outcomes dataset: 0


## Define phenoconversion

In [8]:
# ---------------------------------------------------------------------
# Define phenoconversion
# ---------------------------------------------------------------------

CONVERTER_CASE_TYPES = {
    "persistent_pd",
    "pd_end_of_followup",
}

outcomes["pd_converter"] = (
    outcomes["pd_case_type"]
    .isin(CONVERTER_CASE_TYPES)
    .astype(int)
)

converter_counts = (
    outcomes["pd_converter"]
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Non-converters",
        1: "Converters",
    })
)

print("Phenoconversion outcome:")
print(converter_counts)

print("\nPhenoconversion proportion:")
print(
    f"{outcomes['pd_converter'].mean() * 100:.1f}% "
    "of the at-risk cohort are classified as converters."
)

Phenoconversion outcome:
pd_converter
Non-converters    2085
Converters         162
Name: count, dtype: int64

Phenoconversion proportion:
7.2% of the at-risk cohort are classified as converters.


In [9]:
print("\nPD case type distribution:")
print(
    outcomes["pd_case_type"]
    .value_counts(dropna=False)
)

print("\nPD case type × conversion classification:")
print(
    pd.crosstab(
        outcomes["pd_case_type"],
        outcomes["pd_converter"],
        margins=True,
    )
)


PD case type distribution:
pd_case_type
no_pd                     2061
persistent_pd               88
pd_end_of_followup          74
pd_reversal                 20
pd_competing_diagnosis       4
Name: count, dtype: int64

PD case type × conversion classification:
pd_converter               0    1   All
pd_case_type                           
no_pd                   2061    0  2061
pd_competing_diagnosis     4    0     4
pd_end_of_followup         0   74    74
pd_reversal               20    0    20
persistent_pd              0   88    88
All                     2085  162  2247


## Determine genetic data availability

In [10]:
# ---------------------------------------------------------------------
# Genetic data availability within the at-risk cohort
# ---------------------------------------------------------------------

# For PRS
at_risk_set = set(at_risk_patno)

prs_patno = set(genetics_prs["PATNO"])

at_risk_with_prs = at_risk_set.intersection(prs_patno)

print(
    f"At-risk participants:              {len(at_risk_set):,}"
)
print(
    f"At-risk participants with PRS/PC:   {len(at_risk_with_prs):,}"
)
print(
    f"At-risk participants without PRS/PC: "
    f"{len(at_risk_set - prs_patno):,}"
)

At-risk participants:              2,247
At-risk participants with PRS/PC:   549
At-risk participants without PRS/PC: 1,698


In [11]:
# For variants

variant_patno = set(genetics_variants["PATNO"])

at_risk_with_variants = at_risk_set.intersection(variant_patno)

print(
    f"\nAt-risk participants with variant data: "
    f"{len(at_risk_with_variants):,}"
)
print(
    f"At-risk participants without variant data: "
    f"{len(at_risk_set - variant_patno):,}"
)


At-risk participants with variant data: 1,919
At-risk participants without variant data: 328


## Create the PRS analytic cohort

In [12]:
# ---------------------------------------------------------------------
# PRS/PC data restricted to the at-risk cohort
# ---------------------------------------------------------------------

prs_at_risk = genetics_prs[
    genetics_prs["PATNO"].isin(at_risk_set)
].copy()

print(
    f"PRS/PC records retained for the at-risk cohort: "
    f"{len(prs_at_risk):,}"
)

prs_at_risk = prs_at_risk.merge(
    outcomes[["PATNO", "pd_case_type", "pd_converter"]],
    on="PATNO",
    how="left",
    validate="one_to_one",
)

print(
    f"Final PRS/PC analytic dataset: "
    f"{prs_at_risk.shape[0]:,} participants × "
    f"{prs_at_risk.shape[1]:,} variables"
)

print("\nOutcome distribution:")
print(
    prs_at_risk["pd_converter"]
    .value_counts()
    .rename(index={
        0: "Non-converters",
        1: "Converters",
    })
)

PRS/PC records retained for the at-risk cohort: 549
Final PRS/PC analytic dataset: 549 participants × 111 variables

Outcome distribution:
pd_converter
Non-converters    508
Converters         41
Name: count, dtype: int64


## Ancestry analysis

In [13]:
# ---------------------------------------------------------------------
# Genetic population / ancestry distribution
# ---------------------------------------------------------------------

ancestry_distribution = (
    prs_at_risk["Genetic_PRS_InfPop"]
    .value_counts(dropna=False)
    .rename_axis("Genetic_PRS_InfPop")
    .reset_index(name="count")
)

ancestry_distribution["percentage"] = (
    ancestry_distribution["count"]
    / len(prs_at_risk)
    * 100
)

print("Genetic population distribution in the at-risk genetic cohort:")
print(
    ancestry_distribution.to_string(index=False)
)


ancestry_by_outcome = pd.crosstab(
    prs_at_risk["Genetic_PRS_InfPop"],
    prs_at_risk["pd_converter"],
    margins=True,
)

print("\nGenetic population × phenoconversion:")
print(ancestry_by_outcome)

Genetic population distribution in the at-risk genetic cohort:
Genetic_PRS_InfPop  count  percentage
               EUR    533   97.085610
               AFR     10    1.821494
             OTHER      6    1.092896

Genetic population × phenoconversion:
pd_converter          0   1  All
Genetic_PRS_InfPop              
AFR                   9   1   10
EUR                 494  39  533
OTHER                 5   1    6
All                 508  41  549


## PRS/PC variable discovery

In [14]:
# ---------------------------------------------------------------------
# Identify genetic PRS and PC variables
# ---------------------------------------------------------------------

prs_columns = [
    column
    for column in prs_at_risk.columns
    if "Genetic_PRS_PR" in column
]

pc_columns = [
    column
    for column in prs_at_risk.columns
    if column.startswith("Genetic_PRS_PC")
]

print("PRS variables:")
for column in prs_columns:
    print(f"  - {column}")

print("\nGenetic PC variables:")
for column in pc_columns:
    print(f"  - {column}")

PRS variables:
  - Genetic_PRS_PRS88
  - Genetic_PRS_PRS83
  - Genetic_PRS_PRSp90
  - Genetic_PRS_PRSp88
  - Genetic_PRS_PRSp87
  - Genetic_PRS_PRSp85

Genetic PC variables:
  - Genetic_PRS_PC1
  - Genetic_PRS_PC2
  - Genetic_PRS_PC3
  - Genetic_PRS_PC4
  - Genetic_PRS_PC5
  - Genetic_PRS_PC6
  - Genetic_PRS_PC7
  - Genetic_PRS_PC8
  - Genetic_PRS_PC9
  - Genetic_PRS_PC10


## 6. Individual genetic variant data

The IU genetic consensus file contains one record per PPMI participant.

The PD-related genetic variables of interest are:

- `LRRK2`
- `GBA`
- `VPS35`
- `SNCA`
- `PRKN`
- `PARK7`
- `PINK1`

`APOE` is retained separately because it is not part of the PD-specific
Nalls PRS and has a different biological interpretation.

The original variables are preserved because they contain the reported
variant identity for carriers.

Binary carrier variables will be created as derived variables and will
not overwrite the original PPMI data.

In [20]:
# ---------------------------------------------------------------------
# Genetic variables of interest
# ---------------------------------------------------------------------

PD_GENES = [
    "LRRK2",
    "GBA",
    "VPS35",
    "SNCA",
    "PRKN",
    "PARK7",
    "PINK1",
]

OTHER_GENETIC_VARIABLES = [
    "APOE",
]

print("PD-related genetic variables:")
for gene in PD_GENES:
    print(f"  - {gene}")

print("\nOther genetic variables retained for separate investigation:")
for variable in OTHER_GENETIC_VARIABLES:
    print(f"  - {variable}")

PD-related genetic variables:
  - LRRK2
  - GBA
  - VPS35
  - SNCA
  - PRKN
  - PARK7
  - PINK1

Other genetic variables retained for separate investigation:
  - APOE


In [21]:
# ---------------------------------------------------------------------
# Inspect original genetic variant coding
# ---------------------------------------------------------------------

for gene in PD_GENES:
    print(f"\n{'=' * 60}")
    print(f"{gene}")
    print(f"{'=' * 60}")
    
    print(
        genetics_variants[gene]
        .value_counts(dropna=False)
        .head(20)
    )


LRRK2
LRRK2
0                5241
G2019S            621
NaN               336
R1441G             57
G2019S/G2019S       6
R1441C              1
N1437H              1
I2020T              1
R1441H              1
Name: count, dtype: int64

GBA
GBA
0                     3535
NaN                   2164
N409S                  491
N409S/N409S             31
L483P                   19
L29Afs*18                9
R159W                    4
IVS2+1G>A                4
R535H                    3
R502C                    1
S212X                    1
N409S / c.762-2A>G       1
F255Y                    1
Q112X                    1
Name: count, dtype: int64

VPS35
VPS35
NaN      3251
0        3013
R524Q       1
Name: count, dtype: int64

SNCA
SNCA
NaN            3178
0              3041
A53T             45
duplication       1
Name: count, dtype: int64

PRKN
PRKN
NaN                              3254
0                                2978
R275W                              14
Q34Rfs*5                   

## Restrict variant data to your actual at-risk cohort

In [22]:
# ---------------------------------------------------------------------
# Restrict variant data to the at-risk cohort
# ---------------------------------------------------------------------

variants_at_risk = genetics_variants[
    genetics_variants["PATNO"].isin(at_risk_set)
].copy()

print(
    f"Variant records in full IU genetic dataset: "
    f"{len(genetics_variants):,}"
)

print(
    f"Variant records within at-risk cohort: "
    f"{len(variants_at_risk):,}"
)

print(
    f"Unique at-risk participants with variant records: "
    f"{variants_at_risk['PATNO'].nunique():,}"
)

Variant records in full IU genetic dataset: 6,265
Variant records within at-risk cohort: 1,919
Unique at-risk participants with variant records: 1,919


In [23]:
print(
    f"At-risk participants without a record in the IU genetic dataset: "
    f"{len(at_risk_set - set(variants_at_risk['PATNO'])):,}"
)

At-risk participants without a record in the IU genetic dataset: 328


## Merge the two genetic datasets

In [24]:
# ---------------------------------------------------------------------
# Select genetic variables for analysis
# ---------------------------------------------------------------------

variant_columns = ["PATNO"] + PD_GENES + OTHER_GENETIC_VARIABLES

variants_selected = variants_at_risk[variant_columns].copy()

print(
    f"Selected variant dataset: "
    f"{variants_selected.shape[0]:,} participants × "
    f"{variants_selected.shape[1]:,} variables"
)

Selected variant dataset: 1,919 participants × 9 variables


In [25]:
# ---------------------------------------------------------------------
# Merge PRS/PC data with individual genetic variant data
# ---------------------------------------------------------------------

genetic_data = prs_at_risk.merge(
    variants_selected,
    on="PATNO",
    how="left",
    validate="one_to_one",
)

print(
    f"Combined genetic dataset: "
    f"{genetic_data.shape[0]:,} participants × "
    f"{genetic_data.shape[1]:,} variables"
)

Combined genetic dataset: 549 participants × 119 variables


In [26]:
print(
    f"Participants retained after genetic merge: "
    f"{genetic_data['PATNO'].nunique():,}"
)

print(
    f"Participants missing variant data after merge: "
    f"{genetic_data[PD_GENES].isna().all(axis=1).sum():,}"
)

Participants retained after genetic merge: 549
Participants missing variant data after merge: 0


In [27]:
# ---------------------------------------------------------------------
# Genetic variant availability and diversity
# ---------------------------------------------------------------------

variant_summary = []

for gene in PD_GENES:
    series = genetic_data[gene]
    
    n_total = len(series)
    n_missing = series.isna().sum()
    n_non_carrier = (series == 0).sum()
    n_reported_variant = (
        series.notna() & (series != 0)
    ).sum()
    n_unique_variants = (
        series[
            series.notna() & (series != 0)
        ].nunique()
    )
    
    variant_summary.append({
        "gene": gene,
        "n_total": n_total,
        "n_non_carrier": n_non_carrier,
        "n_reported_variant": n_reported_variant,
        "n_missing": n_missing,
        "n_unique_variants": n_unique_variants,
    })

variant_summary = pd.DataFrame(variant_summary)

variant_summary["carrier_percentage"] = (
    variant_summary["n_reported_variant"]
    / variant_summary["n_total"]
    * 100
)

print("Genetic variant summary in the at-risk cohort:")
print(
    variant_summary.to_string(index=False)
)

Genetic variant summary in the at-risk cohort:
 gene  n_total  n_non_carrier  n_reported_variant  n_missing  n_unique_variants  carrier_percentage
LRRK2      549              0                 549          0                  3               100.0
  GBA      549              0                 549          0                  6               100.0
VPS35      549              0                 549          0                  2               100.0
 SNCA      549              0                 549          0                  2               100.0
 PRKN      549              0                 549          0                  3               100.0
PARK7      549              0                 549          0                  2               100.0
PINK1      549              0                 549          0                  2               100.0


## Create carrier variables

In [28]:
# ---------------------------------------------------------------------
# Create derived binary carrier indicators
# ---------------------------------------------------------------------

for gene in PD_GENES:
    genetic_data[f"{gene}_carrier"] = (
        genetic_data[gene].notna()
        & (genetic_data[gene] != 0)
    ).astype(int)

## Check the carrier counts

In [29]:
# ---------------------------------------------------------------------
# Carrier frequency
# ---------------------------------------------------------------------

carrier_columns = [
    f"{gene}_carrier"
    for gene in PD_GENES
]

carrier_counts = (
    genetic_data[carrier_columns]
    .sum()
    .sort_values(ascending=False)
    .rename("n_carriers")
    .to_frame()
)

carrier_counts["carrier_percentage"] = (
    carrier_counts["n_carriers"]
    / len(genetic_data)
    * 100
)

print("PD-related genetic carrier frequencies:")
print(
    carrier_counts.to_string()
)

PD-related genetic carrier frequencies:
               n_carriers  carrier_percentage
LRRK2_carrier         549               100.0
GBA_carrier           549               100.0
VPS35_carrier         549               100.0
SNCA_carrier          549               100.0
PRKN_carrier          549               100.0
PARK7_carrier         549               100.0
PINK1_carrier         549               100.0


In [30]:
# ---------------------------------------------------------------------
# Carrier status × phenoconversion
# ---------------------------------------------------------------------

for gene in PD_GENES:
    carrier_column = f"{gene}_carrier"
    
    print(f"\n{'=' * 60}")
    print(f"{gene}")
    print(f"{'=' * 60}")
    
    table = pd.crosstab(
        genetic_data[carrier_column],
        genetic_data["pd_converter"],
        margins=True,
    )
    
    print(table)


LRRK2
pd_converter     0   1  All
LRRK2_carrier              
1              508  41  549
All            508  41  549

GBA
pd_converter    0   1  All
GBA_carrier               
1             508  41  549
All           508  41  549

VPS35
pd_converter     0   1  All
VPS35_carrier              
1              508  41  549
All            508  41  549

SNCA
pd_converter    0   1  All
SNCA_carrier              
1             508  41  549
All           508  41  549

PRKN
pd_converter    0   1  All
PRKN_carrier              
1             508  41  549
All           508  41  549

PARK7
pd_converter     0   1  All
PARK7_carrier              
1              508  41  549
All            508  41  549

PINK1
pd_converter     0   1  All
PINK1_carrier              
1              508  41  549
All            508  41  549


## Genetic separation/QC table

In [31]:
# ---------------------------------------------------------------------
# Genetic carrier / phenoconversion QC
# ---------------------------------------------------------------------

genetic_qc = []

for gene in PD_GENES:
    carrier = genetic_data[f"{gene}_carrier"]
    outcome = genetic_data["pd_converter"]
    
    non_carrier_non_converter = (
        ((carrier == 0) & (outcome == 0)).sum()
    )
    
    non_carrier_converter = (
        ((carrier == 0) & (outcome == 1)).sum()
    )
    
    carrier_non_converter = (
        ((carrier == 1) & (outcome == 0)).sum()
    )
    
    carrier_converter = (
        ((carrier == 1) & (outcome == 1)).sum()
    )
    
    n_carriers = (carrier == 1).sum()
    
    if n_carriers > 0:
        carrier_conversion_rate = (
            carrier_converter / n_carriers
        )
    else:
        carrier_conversion_rate = np.nan
    
    # Flag zero cells in the carrier outcome group.
    separation_warning = (
        n_carriers > 0
        and (
            carrier_converter == 0
            or carrier_non_converter == 0
        )
    )
    
    genetic_qc.append({
        "gene": gene,
        "n_carriers": n_carriers,
        "carrier_converter": carrier_converter,
        "carrier_non_converter": carrier_non_converter,
        "carrier_conversion_rate": carrier_conversion_rate,
        "separation_warning": separation_warning,
    })

genetic_qc = pd.DataFrame(genetic_qc)

print("Genetic carrier/outcome QC:")
print(
    genetic_qc.to_string(index=False)
)

Genetic carrier/outcome QC:
 gene  n_carriers  carrier_converter  carrier_non_converter  carrier_conversion_rate  separation_warning
LRRK2         549                 41                    508                 0.074681               False
  GBA         549                 41                    508                 0.074681               False
VPS35         549                 41                    508                 0.074681               False
 SNCA         549                 41                    508                 0.074681               False
 PRKN         549                 41                    508                 0.074681               False
PARK7         549                 41                    508                 0.074681               False
PINK1         549                 41                    508                 0.074681               False


In [32]:
print(genetic_data["pd_converter"].value_counts(dropna=False))

pd_converter
0    508
1     41
Name: count, dtype: int64


## 6. Individual genetic variant investigation

For each PD-related gene, investigate the original PPMI genetic coding
without binarisation.

Each genetic variable is examined according to:

- non-carrier (`0`)
- reported variant(s)
- missing (`NaN`)

These categories are cross-tabulated against phenoconversion status
(converter vs non-converter).

In [33]:
PD_GENES = [
    "LRRK2",
    "GBA",
    "VPS35",
    "SNCA",
    "PRKN",
    "PARK7",
    "PINK1",
]

print(f"Genes to investigate: {len(PD_GENES)}")
print(", ".join(PD_GENES))

Genes to investigate: 7
LRRK2, GBA, VPS35, SNCA, PRKN, PARK7, PINK1


In [34]:
for gene in PD_GENES:
    print(f"\n{'=' * 70}")
    print(f"{gene} — original PPMI coding")
    print(f"{'=' * 70}")
    
    print(
        genetic_data[gene]
        .value_counts(dropna=False)
        .to_string()
    )


LRRK2 — original PPMI coding
LRRK2
0         393
G2019S    144
R1441G     12

GBA — original PPMI coding
GBA
0              396
N409S          139
N409S/N409S      9
L483P            3
R535H            1
L29Afs*18        1

VPS35 — original PPMI coding
VPS35
0        548
R524Q      1

SNCA — original PPMI coding
SNCA
0       542
A53T      7

PRKN — original PPMI coding
PRKN
0                                546
R275W                              2
<DUP>chr6:162423548-162571629      1

PARK7 — original PPMI coding
PARK7
0        548
P127S      1

PINK1 — original PPMI coding
PINK1
0        548
R492X      1


In [38]:
for gene in PD_GENES:
    gene_for_analysis = (
        genetic_data[gene]
        .astype("object")
        .where(
            genetic_data[gene].notna(),
            "MISSING",
        )
    )
    
    summary = (
        pd.crosstab(
            gene_for_analysis,
            genetic_data["pd_converter"],
        )
        .rename(
            columns={
                0: "Non-converters",
                1: "Converters",
            }
        )
    )
    
    summary["Total"] = summary.sum(axis=1)
    
    summary["Conversion_rate"] = (
        summary["Converters"]
        / summary["Total"]
        * 100
    )
    
    print(f"\n{'=' * 70}")
    print(f"{gene} — genetic status × phenoconversion")
    print(f"{'=' * 70}")
    print(summary.to_string())


LRRK2 — genetic status × phenoconversion
pd_converter  Non-converters  Converters  Total  Conversion_rate
LRRK2                                                           
0                        367          26    393         6.615776
G2019S                   130          14    144         9.722222
R1441G                    11           1     12         8.333333

GBA — genetic status × phenoconversion
pd_converter  Non-converters  Converters  Total  Conversion_rate
GBA                                                             
0                        361          35    396         8.838384
L29Afs*18                  1           0      1         0.000000
L483P                      2           1      3        33.333333
N409S                    135           4    139         2.877698
N409S/N409S                8           1      9        11.111111
R535H                      1           0      1         0.000000

VPS35 — genetic status × phenoconversion
pd_converter  Non-converters  C

In [40]:
PD_GENES = ["LRRK2", "GBA", "VPS35", "SNCA", "PRKN", "PARK7", "PINK1"]

variant_summary = []

for gene in PD_GENES:
    counts = genetic_data[gene].value_counts(dropna=False)

    n_carriers = genetic_data[gene].notna().sum() - (genetic_data[gene] == 0).sum()
    n_non_carriers = (genetic_data[gene] == 0).sum()
    n_missing = genetic_data[gene].isna().sum()

    variant_summary.append({
        "Gene": gene,
        "Non-carriers": n_non_carriers,
        "Carriers": n_carriers,
        "Missing": n_missing,
        "Unique_variant_values": genetic_data.loc[
            genetic_data[gene].notna() & (genetic_data[gene] != 0), gene
        ].nunique()
    })

variant_summary = pd.DataFrame(variant_summary)

print(variant_summary.to_string(index=False))

 Gene  Non-carriers  Carriers  Missing  Unique_variant_values
LRRK2             0       549        0                      3
  GBA             0       549        0                      6
VPS35             0       549        0                      2
 SNCA             0       549        0                      2
 PRKN             0       549        0                      3
PARK7             0       549        0                      2
PINK1             0       549        0                      2


In [41]:
genetic_data["LRRK2_carrier"] = genetic_data["LRRK2"].notna() & (
    genetic_data["LRRK2"] != 0
)

genetic_data["GBA_carrier"] = genetic_data["GBA"].notna() & (
    genetic_data["GBA"] != 0
)

genetic_data["LRRK2_carrier"] = genetic_data["LRRK2_carrier"].astype(int)
genetic_data["GBA_carrier"] = genetic_data["GBA_carrier"].astype(int)

In [42]:
genetic_data

,PATNO,EVENT_ID,Genetic_PRS_InfPop,Genetic_PRS_PRS88,Genetic_PRS_PRS83,Genetic_PRS_PRSp90,Genetic_PRS_PRSp88,Genetic_PRS_PRSp87,Genetic_PRS_PRSp85,Genetic_PRS_PC1,...,PARK7,PINK1,APOE,LRRK2_carrier,GBA_carrier,VPS35_carrier,SNCA_carrier,PRKN_carrier,PARK7_carrier,PINK1_carrier
0,12224,SC,EUR,-0.571687,-0.451707,-0.511607,-0.515509,-0.381914,-0.334952,-0.567199,...,0,0,E3/E4,1,1,1,1,1,1,1
1,12593,SC,EUR,-0.085392,0.924425,-0.144565,0.497992,-0.012014,0.811158,0.214098,...,0,0,E3/E4,1,1,1,1,1,1,1
2,13039,SC,EUR,0.114803,1.301002,0.116341,0.929356,0.250921,1.298970,0.077131,...,0,0,E3/E3,1,1,1,1,1,1,1
3,14281,SC,EUR,-0.861197,-0.534909,-0.862083,-0.688287,-0.735107,-0.530364,-0.329822,...,0,0,E3/E4,1,1,1,1,1,1,1
4,14331,SC,EUR,-0.678230,-0.190715,-0.618439,-0.285485,-0.489569,-0.074827,-0.386085,...,0,0,E2/E3,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544,60108,SC,EUR,-0.514328,0.117567,-0.514514,-0.113652,-0.384843,0.119465,-0.198595,...,0,0,E3/E3,1,1,1,1,1,1,1
545,60109,SC,EUR,-1.162557,-1.101778,-1.164520,-1.188321,-1.039899,-1.095831,-0.428923,...,0,0,E3/E3,1,1,1,1,1,1,1
546,60170,SC,AFR,-1.518726,-1.771728,-1.400812,-1.579013,-1.278030,-1.537626,6.191919,...,0,0,E3/E3,1,1,1,1,1,1,1
547,85062,SC,EUR,-0.806403,-0.882627,-0.807383,-0.597870,-0.922167,-0.877410,-0.003101,...,0,0,E3/E3,1,1,1,1,1,1,1
